# Task B Run 23 -- Gemma-4-12B: four epochs vs three, then a four-model submission

Both of Run 21's seeds were still rising at the last epoch (0.597, 0.629, 0.657 and 0.561,
0.630, 0.661), and averaging the two seeds added about 2 points.

**Stage 1: four epochs on the holdout.** Seeds 42 and 43 run in parallel, about 1.6 h.
- **Without an attached Run 21 output,** the comparison is with Run 21's recorded 3-epoch
  two-seed score, **0.6792**, on the same rows and seeds.
- **With it attached,** Run 21's 3-epoch probabilities are also loaded. That allows a
  paired bootstrap and a third candidate, `mix`, which averages all four models.

**Stage 2: four full-data models of the winner, averaged.** Two GPUs run two at a time, so
four 4-epoch seeds take about 4 h. Run 21's 3-epoch full fits are reused when attached and
relevant. The output is `RECOMMENDED_b23_gemma.zip`.

**Settings:** GPU T4 x2, Internet on. No token and no attached input are needed.
Save & Run All. A hard stop at 11 h means the notebook always finishes and packages
what it has.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch
T_START = time.time()
HARD_STOP = T_START + 11 * 3600   # Kaggle kills at 12 h and then saves nothing

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/tmp/hastika"             # outside /kaggle/working: the output holds results only


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Settings -> Internet, then restart.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                              # Gemma-4 is not gated; no token is needed

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path("/kaggle/working/b23_outputs")
RUNS, LOGS, SUB = OUT / "runs", OUT / "logs", OUT / "submissions"
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

TASK = "b"
CLASSES = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
LABEL_COL = "Hate Category"
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(df), len(holdout)) == (3532, 530), (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(CLASSES.index).to_numpy()
TEST_CSV = pathlib.Path("/tmp/test_inputs.csv")
pd.read_csv(pathlib.Path(WORK) / "data/raw/hastika_multiclass_test.csv")[["id", "Comment"]].to_csv(TEST_CSV, index=False)
test = pd.read_csv(TEST_CSV)
EXTERNAL = pathlib.Path(WORK) / "data/external/offenseval_kn.csv"
print(f"task {TASK.upper()}: train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), "
      f"all {len(df)}, test {len(test)}")


def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    macro = float(f1_score(y, pred, average="macro"))
    print(f"{name:34s} macro-F1 {macro:.4f}  acc {accuracy_score(y, pred):.4f}  | "
          + "  ".join(f"{c[:5]} {f:.3f}" for c, f in zip(CLASSES, per)))
    return macro


def boot_line(label, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f"{label:28s} {r['diff']:+.4f}  95% CI [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]  "
          f"P(better) {r['p_better']:.2f}")
    return r


# Earlier runs' outputs are reused when attached (Add Input -> Notebook Output), found by
# walking every input with symlinks followed. Nothing below *requires* them.
def find_prior(pattern):
    hits = []
    for root, _, files in os.walk("/kaggle/input", followlinks=True):
        for f in files:
            p = os.path.join(root, f)
            if fnmatch(p, pattern):
                hits.append(pathlib.Path(p))
    return sorted(hits)


def show_inputs():
    if not os.path.isdir("/kaggle/input"):
        print("no /kaggle/input")
        return
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 4:
            print("  " * depth + (os.path.basename(root) or "/kaggle/input") + f"/  ({len(files)} files)")


print("attached inputs:")
show_inputs()

In [ ]:
# LLM stack in its own environment (transformers >= 5.12 for Gemma-4); Kaggle's torch and
# CUDA are reused via --system-site-packages. The repo's MuRIL/SVM code stays on the
# system environment.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import transformers, peft, bitsandbytes; print('LLM env: "
                "transformers', transformers.__version__, '| peft', peft.__version__)"], check=True)

MODEL = "google/gemma-4-12B"
S = "gemma-4-12b"


def download(model=MODEL):
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])


def llm_job(name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def tapt_job(name, texts):
    """Gemma TAPT: LoRA next-token training on unlabelled comments (the MuRIL +2.9 lesson)."""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} "
                    f"--texts {' '.join(map(str, texts))} --out {RUNS / name} --cache {HF_CACHE}")}


def load_probs(run_dir, csv):
    """<input stem>_probs.npy, or None when it is missing, belongs to other rows (another
    task's output found among the inputs), or is collapsed onto one class (Task A Run 28's
    seed 43 predicted Non-Hate for every row)."""
    run_dir, csv = pathlib.Path(run_dir), pathlib.Path(csv)
    p = run_dir / f"{csv.stem}_probs.npy"
    if not p.exists():
        return None
    ids = pd.read_csv(run_dir / f"{csv.stem}_ids.csv")["id"].tolist()
    if ids != pd.read_csv(csv)["id"].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=probs.shape[1]) / len(probs)
    if share.max() > 0.9:
        print(f"skipping {run_dir.name}: collapsed, {share.max():.0%} of rows in one class")
        return None
    return probs


def run_jobs(jobs, on_done=None):
    """run_queue with two automatic retries per job: exit code 4 (collapsed after epoch 1)
    reruns with seed + 1000; out of GPU memory reruns at micro-batch 4 (same effective
    batch). on_done sees each job's final outcome."""
    import re
    registry, retried = {j["name"]: j for j in jobs}, set()

    def done(name, code):
        job, nxt = registry.get(name), []
        if code and job is not None and name not in retried:
            log = open(job["log"], errors="replace").read()
            seed = re.search(r"--seed (\d+)", job["cmd"])
            if code == 4 and seed:
                print(f"{name} collapsed -- rerunning with seed {int(seed.group(1)) + 1000}")
                nxt = [{**job, "cmd": job["cmd"] + f" --seed {int(seed.group(1)) + 1000}"}]
            elif "OutOfMemoryError" in log and "llm_tapt" in job["cmd"]:
                # TAPT: same effective batch (16) at micro-batch 1, smaller logit chunks
                print(f"{name} ran out of GPU memory -- rerunning TAPT at micro-batch 1")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 1 --grad-accum 16 --logit-chunk 16"}]
            elif "OutOfMemoryError" in log:
                print(f"{name} ran out of GPU memory -- rerunning at micro-batch 4")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"}]
            if nxt:
                retried.add(name)
        if not nxt and on_done is not None:
            nxt = list(on_done(name, code) or [])
        for j in nxt:
            registry[j["name"]] = j
        return nxt

    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=done, stop_at=HARD_STOP)


def write_zip(name, probs, recommended=False):
    import zipfile
    assert probs.shape == (len(test), len(CLASSES)), probs.shape
    pred = pd.DataFrame({"id": test["id"], "label": [CLASSES[i] for i in probs.argmax(1)]})
    assert pred["id"].is_unique and set(pred["label"]) <= set(CLASSES)
    np.save(SUB / f"{name}_test_probs.npy", probs)
    csv = SUB / f"predictions_{name}.csv"
    pred.to_csv(csv, index=False)
    path = SUB / f"{'RECOMMENDED_' if recommended else ''}{name}.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, "predictions.csv")
    prior = df[LABEL_COL].value_counts(normalize=True)
    counts = pred["label"].value_counts(normalize=True)
    print(f"{path.name}  predicted/prior: "
          + "  ".join(f"{c[:5]} {counts.get(c, 0):.1%}/{prior[c]:.1%}" for c in CLASSES))
    return path


download()

In [ ]:
RUN21_3EP = 0.6792                 # Run 21, 3 epochs, seeds 42+43, this holdout
ho3, full3 = {}, {}
for s in (42, 43):
    h = find_prior(f"*/runs/{S}_ho_s{s}/holdout_probs.npy")
    if h:
        ho3[s] = load_probs(h[0].parent, SPLIT / "holdout.csv")
    f = find_prior(f"*/runs/{S}_full_s{s}/test_inputs_probs.npy")
    if f:
        full3[s] = load_probs(f[0].parent, TEST_CSV)
ho3 = {k: v for k, v in ho3.items() if v is not None}
full3 = {k: v for k, v in full3.items() if v is not None}
print("reused from Run 21 -- holdout seeds:", list(ho3), "| full-fit seeds:", list(full3))

jobs = [llm_job(f"{S}_ep4_ho_s{s}", SPLIT / "train.csv", [SPLIT / "holdout.csv"], s,
                evalf=SPLIT / "holdout.csv", extra="--epochs 4") for s in (42, 43)]
codes1 = run_jobs(jobs)
ho4 = {s: load_probs(RUNS / f"{S}_ep4_ho_s{s}", SPLIT / "holdout.csv") for s in (42, 43)}
ho4 = {k: v for k, v in ho4.items() if v is not None}
for s in ho4:
    hist = json.load(open(RUNS / f"{S}_ep4_ho_s{s}" / "metrics.json"))["history"]
    print(f"4 epochs seed {s} per epoch (diagnostic):", [round(h["eval_macro_f1"], 4) for h in hist])

In [ ]:
scores = {"3ep": report(y_ho, np.mean(list(ho3.values()), 0), "3 epochs (Run 21)")
          if len(ho3) == 2 else RUN21_3EP}
if len(ho3) != 2:
    print(f"3 epochs (Run 21, recorded)          macro-F1 {RUN21_3EP:.4f}")
boots = {}
if len(ho4) == 2:
    p4 = np.mean(list(ho4.values()), 0)
    scores["4ep"] = report(y_ho, p4, "4 epochs, 2 seeds")
    if len(ho3) == 2:
        p3 = np.mean(list(ho3.values()), 0)
        pm = np.mean(list(ho3.values()) + list(ho4.values()), 0)
        scores["mix"] = report(y_ho, pm, "3 + 4 epochs, 4 models")
        boots["4ep - 3ep"] = boot_line("4ep - 3ep", p4, p3)
        boots["mix - 3ep"] = boot_line("mix - 3ep", pm, p3)
CHOICE = max(scores, key=scores.get)
print("\nchosen:", CHOICE, scores)

In [ ]:
def full(seed, ep4):
    return llm_job(f"{S}_{'ep4_' if ep4 else ''}full_s{seed}", SPLIT / "all.csv", [TEST_CSV],
                   seed, extra="--epochs 4" if ep4 else "")


if CHOICE == "3ep":
    plan = [full(s, False) for s in (42, 43, 44, 45) if s not in full3]
elif CHOICE == "4ep":
    plan = [full(s, True) for s in (42, 43, 44, 45)]
else:                               # mix: Run 21's 3-epoch fits plus two 4-epoch fits
    plan = [full(s, True) for s in (42, 43)]
print("queueing:", [j["name"] for j in plan])
codes2 = run_jobs(plan)

members = {}
if CHOICE in ("3ep", "mix"):
    members.update({f"run21_3ep_s{s}": p for s, p in full3.items()})
for j in plan:
    p = load_probs(RUNS / j["name"], TEST_CSV)
    if p is not None:
        members[j["name"]] = p
assert members, "no full fit finished -- see the logs"
print("averaging", len(members), "models:", list(members))
write_zip("b23_gemma", np.mean(list(members.values()), 0), recommended=True)
json.dump({"commit": head, "choice": CHOICE, "holdout_scores": scores, "bootstrap": boots,
           "members": list(members), "exit_codes": {**codes1, **codes2}},
          open(OUT / "result.json", "w"), indent=2)
print(f"{(time.time() - T_START) / 3600:.1f} h")